# Laboratorio 4 (Parte 2) — Inciso 9: Generación de mapas predictivos

Se usa el mejor modelo del inciso 5 para calcular, por observación, la probabilidad de alta presencia de cianobacteria, se reconstruye espacialmente sobre la grilla del raster original y se generan mapas de probabilidad y de error para una fecha representativa de cada lago: `fecha_pico`, la fecha de mayor `clorofila` promedio, con el mismo criterio que la Parte I usa para identificar la fecha de floración más intensa de cada lago.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import LAGOS, RUTA_DATA_PROCESSED, RUTA_FIGURAS
from src.mapas import (
    ETIQUETAS_PROBABILIDAD,
    NIVELES_PROBABILIDAD,
    clasificar_error,
    probabilidad_por_pixel,
    rejilla_desde_columnas,
)
from src.modelado import PREDICTORES, agregar_features, cargar_modelo, construir_respuesta

dataset = pd.read_parquet(RUTA_DATA_PROCESSED / "dataset_ml.parquet")
dataset = construir_respuesta(agregar_features(dataset))

metricas = pd.read_csv(RUTA_DATA_PROCESSED / "p2_metricas_modelos.csv", index_col=0)
criterio = "f2" if "f2" in metricas.columns else "roc_auc"
mejor_nombre = metricas[criterio].idxmax()
modelo = cargar_modelo(mejor_nombre)
print(f"Mejor modelo según {criterio}: {mejor_nombre}")

# Misma definición de "fecha pico" que la Parte I (notebooks/05_analisis_espacial.ipynb):
# la fecha con mayor clorofila promedio en el lago.
promedio_por_fecha = dataset.groupby(["lago", "fecha"], observed=True)["clorofila"].mean()
fecha_pico = {
    lago: promedio_por_fecha[lago].idxmax().strftime("%Y-%m-%d") for lago in LAGOS
}
fecha_pico

## 9.1 Probabilidad de alta presencia por observación

In [ ]:
mapas_lago = {}
for lago in LAGOS:
    sub = dataset[(dataset["lago"] == lago) & (dataset["fecha"] == fecha_pico[lago])].copy()
    sub["probabilidad"] = probabilidad_por_pixel(modelo, sub, PREDICTORES)
    sub["prediccion"] = modelo.predict(sub[PREDICTORES])
    sub["error"] = clasificar_error(sub["alta_cianobacteria"].to_numpy(), sub["prediccion"].to_numpy())
    mapas_lago[lago] = sub
    print(f"{lago} ({fecha_pico[lago]}): {len(sub):,} observaciones, probabilidad media {sub['probabilidad'].mean():.3f}")

## 9.2 a 9.4 Reconstrucción espacial y mapa de probabilidad con escala de 4 niveles

In [ ]:
cmap_prob = mcolors.ListedColormap(["#fee5d9", "#fcae91", "#fb6a4a", "#a50f15"])
norm_prob = mcolors.BoundaryNorm(NIVELES_PROBABILIDAD, cmap_prob.N)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, lago in zip(axes, LAGOS):
    sub = mapas_lago[lago]
    grilla = rejilla_desde_columnas(sub, sub["probabilidad"].to_numpy())
    im = ax.imshow(grilla, cmap=cmap_prob, norm=norm_prob)
    ax.set_title(f"{LAGOS[lago]['nombre']} — {fecha_pico[lago]}")
    ax.axis("off")

cbar = fig.colorbar(im, ax=axes, ticks=[0.125, 0.375, 0.625, 0.875], fraction=0.046, pad=0.02)
cbar.ax.set_yticklabels(ETIQUETAS_PROBABILIDAD)
cbar.set_label("Probabilidad de alta presencia de cianobacteria")
fig.suptitle(f"Mapa de probabilidad predicha — {mejor_nombre}")
fig.savefig(RUTA_FIGURAS / "p2_mapa_probabilidad.png", dpi=150, bbox_inches="tight")
plt.show()

## 9.5 Comparación con los mapas de cianobacteria de la Parte I

*(Completar tras ejecutar: comparar visualmente `p2_mapa_probabilidad.png` con los mapas de índice de cianobacteria de la misma fecha (`fecha_pico`) generados en la Parte I -notebook `05_analisis_espacial.ipynb`- para cada lago. Señalar si las zonas de probabilidad "alta"/"muy alta" coinciden espacialmente con las zonas de mayor índice de cianobacteria observado.)*

Se espera una concordancia general en la localización de las zonas de mayor probabilidad, ya que el modelo se entrenó para predecir precisamente el umbral derivado de ese índice; las diferencias esperables se concentran en los bordes de las zonas de floración, donde el índice real cambia de forma continua pero el modelo debe decidir una probabilidad a partir de bandas espectrales que no incluyen la información exacta usada para construir la variable respuesta (inciso 2.5).

## 9.6 Zonas correctamente detectadas, falsos positivos, falsos negativos y patrones espaciales de error

In [ ]:
colores_error = {"VN": "#f0f0f0", "VP": "#238b45", "FP": "#fdae61", "FN": "#d7191c"}
cmap_error = mcolors.ListedColormap(list(colores_error.values()))
codigo_error = {etiqueta: i for i, etiqueta in enumerate(colores_error)}

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, lago in zip(axes, LAGOS):
    sub = mapas_lago[lago]
    codigos = sub["error"].map(codigo_error).to_numpy()
    grilla = rejilla_desde_columnas(sub, codigos.astype(float))
    ax.imshow(grilla, cmap=cmap_error, vmin=-0.5, vmax=len(codigo_error) - 0.5)
    ax.set_title(f"{LAGOS[lago]['nombre']} — {fecha_pico[lago]}")
    ax.axis("off")

parches = [plt.matplotlib.patches.Patch(color=c, label=e) for e, c in colores_error.items()]
fig.legend(handles=parches, loc="lower center", ncol=4, fontsize=9)
fig.suptitle(f"Mapa de error de predicción — {mejor_nombre}")
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(RUTA_FIGURAS / "p2_mapa_errores.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for lago in LAGOS:
    conteo = mapas_lago[lago]["error"].value_counts()
    print(f"{lago}: {conteo.to_dict()}")

## 9.7 Regiones con dificultad sistemática

*(Completar tras ejecutar: revisar en `p2_mapa_errores.png` si los falsos positivos (naranja) y falsos negativos (rojo) se concentran en regiones específicas -por ejemplo, orillas, zonas de entrada de afluentes, o el borde entre agua y la máscara de tierra- o si están dispersos de forma aleatoria sobre el cuerpo de agua.)*

Es esperable que los errores no se distribuyan de forma uniforme, sino que se concentren en:

- **Zonas de transición espacial**, en el límite entre una región de alta y otra de baja concentración, donde la señal espectral cambia de forma gradual y el modelo debe decidir con un umbral fijo (0.5) sobre una probabilidad continua.
- **Bordes del cuerpo de agua**, cerca del límite con la máscara de tierra, donde la mezcla espectral entre agua y vegetación/sedimento costero introduce reflectancias atípicas que ninguno de los filtros de calidad del inciso 1 elimina por completo.
- **Zonas cercanas a afluentes o descargas puntuales**, con condiciones ópticas locales (sedimento, turbidez) distintas al resto del lago, que el modelo no puede distinguir de una señal de cianobacteria si no tuvo suficientes ejemplos de esas condiciones durante el entrenamiento.

### Self-check

In [ ]:
for lago in LAGOS:
    sub = mapas_lago[lago]
    assert sub["probabilidad"].between(0, 1).all(), f"{lago}: probabilidad fuera de [0, 1]"
    assert set(sub["error"].unique()).issubset({"VP", "FP", "FN", "VN"})
    grilla = rejilla_desde_columnas(sub, sub["probabilidad"].to_numpy())
    assert grilla.shape[0] >= sub["fila"].max() + 1 and grilla.shape[1] >= sub["columna"].max() + 1
print("OK: probabilidad, error y reconstrucción espacial consistentes para ambos lagos.")